## 0.0 BaseLine


In [14]:
import pandas as pd

In [16]:
df_raw = pd.read_csv('dataset/employee_attrition_train.csv')
baseline_df = df_raw.copy()

In [17]:
baseline_df.T

,0,1,2,3,4,5,6,7,8,9,...,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028
Age,50.0,36.0,21.0,50.0,52.0,33.0,47.0,22.0,NaN,33.0,...,23.0,41.0,33.0,49.0,33.0,NaN,41.0,22.0,29.0,50.0
Attrition,No,No,Yes,No,No,Yes,No,No,Yes,No,...,No,No,No,No,No,No,No,Yes,No,No
BusinessTravel,Travel_Rarely,Travel_Rarely,Travel_Rarely,Travel_Frequently,Travel_Rarely,Travel_Rarely,Travel_Rarely,Travel_Rarely,Travel_Rarely,Non-Travel,...,Travel_Rarely,Travel_Rarely,Non-Travel,Travel_Rarely,Travel_Rarely,Travel_Rarely,Travel_Rarely,Travel_Frequently,Travel_Rarely,Travel_Rarely
DailyRate,1126.0,216.0,337.0,1246.0,994.0,1277.0,1001.0,1230.0,890.0,530.0,...,160.0,1276.0,750.0,1495.0,589.0,750.0,447.0,1256.0,1378.0,264.0
Department,Research & Development,Research & Development,Sales,Human Resources,Research & Development,Research & Development,Research & Development,Research & Development,Research & Development,Sales,...,Research & Development,Sales,Sales,Research & Development,Research & Development,Research & Development,Research & Development,Research & Development,Research & Development,Sales
DistanceFromHome,1.0,6.0,7.0,NaN,7.0,15.0,4.0,1.0,2.0,16.0,...,4.0,2.0,22.0,NaN,28.0,28.0,NaN,NaN,13.0,9.0
Education,2,2,1,3,4,1,3,2,4,3,...,1,5,2,4,4,3,3,4,2,3
EducationField,Medical,Medical,Marketing,Medical,Life Sciences,Medical,Life Sciences,Life Sciences,Medical,Life Sciences,...,Medical,Life Sciences,Marketing,Technical Degree,Life Sciences,Life Sciences,Life Sciences,Life Sciences,Other,Marketing
EmployeeCount,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
EmployeeNumber,997,178,1780,644,1118,582,1827,872,828,1681,...,1735,625,160,1473,1549,1596,1814,1203,2053,1591


In [ ]:
baseline_df['Annual_Salary'] = baseline_df['MonthlyIncome'] * 12 

# Cálculo 
# Salário anual * percentual de perda (nível do profissional)
# Entry-level : 40% do salário anual de custo
# Professional : 100% salario anual de custo 
# Manager : 200% salario anual de custo

baseline_df['Replacement_Multiplier'] = baseline_df['JobLevel'].apply(lambda x: 0.40 if x<= 2 else  1.00 if x == 3 else 2.00) # Na coluna JobLevel aplicamos um lambda x que percorre todas as linhas da coluna , para calcular o percentual de custo de acordo com o nivel do funcionário , guardamos o resultado em uma nova coluna 
baseline_df['Replacement_Cost'] = baseline_df['Annual_Salary'] * baseline_df['Replacement_Multiplier'] # fazemos o salário anual de cada funcionario multiplicado o percentual de custo dele , para ver quanto cada funcionário causa de prejuízo a empresa cada seja demitido 

# Custo calculado só para as pessoas com Attrition = True 

leaving_df =baseline_df[baseline_df['Attrition']== 'Yes']

total_employee = len(baseline_df) # qtd total de funcionários
leaving_employee = len(leaving_df) # qtd de funcionários que sairam 
attrition_rate = (leaving_employee/total_employee) * 100 # porcentagem da rotatividade de funcionários
avg_replacement_cost = leaving_df['Replacement_Cost'].mean() # média de custo dos funcionários que sairam 
total_annual_cost = leaving_df['Replacement_Cost'].sum() # gasto total anual com rotatividade de funcionários 


print("Situação atual do custo da empresa por rotatividade ")
print(f"Total de funcionários : {total_employee} " )
print(f"Funcionários que saem por ano : {leaving_employee} ")
print(f"Faixa de rotatividade Anual: {attrition_rate :.2f} %")
print(f"Custo médio da rotatividade de funcionários : {avg_replacement_cost :.2f}")
print(f"Custo Total anual da rotatividade de funcionários :  {total_annual_cost :.2f}")


Situação atual do custo da empresa por rotatividade 
Total de funcionários : 1029 
Funcionários que saem por ano : 176 
Faixa de rotatividade Anual: 17.10 %
Custo médio da rotatividade de funcionários : 43870.24
Custo Total anual da rotatividade de funcionários :  7721162.40


In [ ]:
# Simulação 
def roi_simulation (recall , retention_rate):

    # constants
    maintenance_annual_cost = 2000 # custo de manutenção do projeto 
    intervention_cost = 5000   # custo de intervenção do rh sobre o funcionário
    production_cost = 20000 # custo de produção do projeto/horas trabalhadas do cientista

    # pessoas que o algoritmo apontou como potencial de saída 
    true_positive = int(leaving_employee * recall) # cálculo de funcionários salvos , que voce conseguiu reter 
    true_positive # apresentação do resultado 

    # desempenho do RH 
    retention_employee = int(retention_rate * true_positive) 
    avg_savings = retention_employee * avg_replacement_cost

    #print(avg_savings)
    # ROI = retorno / investimento 

    annual_cost = maintenance_annual_cost + (intervention_cost * retention_employee) + production_cost
    liquid_savings = avg_savings - annual_cost
    roi_annual = (avg_savings / annual_cost) 

    return { 
            "Cenário": "Simulado" , 
            "Recall" : recall ,
            "Retencao" : retention_rate, 
            "func_retidos" :int( true_positive * retention_rate) , 
            "Economia.Liquida" : liquid_savings , 
            "Roi Anual " : roi_annualx
    }           
tabela_cenarios = [
roi_simulation(recall = 0.10 , retention_rate= 0.30 ) ,
roi_simulation(recall = 0.40 , retention_rate= 0.30 ) ,
roi_simulation(recall = 0.65 , retention_rate= 0.30 ) ,
roi_simulation(recall = 0.79 , retention_rate= 0.30 ) ,
roi_simulation(recall = 0.90 , retention_rate= 0.30 ) 
]

df_potencial = pd.DataFrame(tabela_cenarios)
df_potencial = df_potencial.style.format({"Economia.Liquida": "${: ,.2f}" })
display(df_potencial)




,Cenário,Recall,Retencao,func_retidos,Economia.Liquida,Roi Anual
0,Simulado,0.100000,0.300000,15,"$ 578,000.00",6.958763
1,Simulado,0.400000,0.300000,60,"$ 2,378,000.00",8.385093
2,Simulado,0.650000,0.300000,97,"$ 3,858,000.00",8.609467
3,Simulado,0.790000,0.300000,118,"$ 4,698,000.00",8.676471
4,Simulado,0.900000,0.300000,135,"$ 5,378,000.00",8.715925
